# 02 · Dataset — smoltalk & Chat Templates

**Runs on: Local CPU**

By the end of this notebook you will understand:
- How instruction datasets are structured (list of messages)
- What `apply_chat_template` produces and why it matters for training
- How to tokenize a sample and inspect sequence lengths
- How to choose `max_seq_length` for fine-tuning

In [ ]:
%pip install datasets transformers matplotlib

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

MODEL_ID = "HuggingFaceTB/SmolLM2-135M"
DATASET_ID = "HuggingFaceTB/smoltalk"
DATASET_CONFIG = "everyday-conversations"  # the subset used to train SmolLM2-Instruct

## 1. Load the Dataset

In [ ]:
# Streaming=True avoids downloading all ~1GB; we'll take a small slice
ds_stream = load_dataset(DATASET_ID, DATASET_CONFIG, split="train", streaming=True)
samples = list(ds_stream.take(3000))  # grab 3k samples for exploration

print(f"Loaded {len(samples)} samples")
print(f"Keys in each sample: {list(samples[0].keys())}")

## 2. Inspect the Data Format

Most instruction datasets use the **chat format**: a list of turns, each with `role` and `content`.  
Roles are typically `system`, `user`, and `assistant`.

In [ ]:
sample = samples[0]
print("Raw sample (first entry):")
print(f"  Type of 'messages': {type(sample['messages'])}")
print(f"  Number of turns:    {len(sample['messages'])}")
print()
for turn in sample["messages"]:
    role = turn["role"]
    content = turn["content"][:200]  # truncate for display
    print(f"  [{role.upper()}]")
    print(f"  {content}{'...' if len(turn['content']) > 200 else ''}")
    print()

In [ ]:
# What's the distribution of roles?
from collections import Counter
role_counts = Counter(turn["role"] for s in samples for turn in s["messages"])
print("Role distribution across 3k samples:")
for role, count in role_counts.most_common():
    print(f"  {role:12s}: {count}")

## 3. Apply the Chat Template

Before tokenizing for training, we serialize the list of messages into a single string using **special tokens**.  
This is what `tokenizer.apply_chat_template()` does.

The model learns to recognize `<|im_start|>user` as "a user turn begins" and `<|im_end|>` as "turn ends".

In [ ]:
CHATML = (
    "{% for message in messages %}"
    "{{'<|im_start|>' + message['role'] + '\\n' + message['content'] + '<|im_end|>' + '\\n'}}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\\n' }}{% endif %}"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Apply the chat template - produces a single string with special tokens
formatted = tokenizer.apply_chat_template(
    sample["messages"],
    chat_template=CHATML,
    tokenize=False
)

print("Formatted text (raw repr showing special tokens):")
print(repr(formatted[:400]))
print()
print("Rendered:")
print(formatted[:600])

In [ ]:
# What special tokens does this tokenizer use?
print("Special tokens:")
print(f"  BOS (begin of sequence): {tokenizer.bos_token!r} (id={tokenizer.bos_token_id})")
print(f"  EOS (end of sequence):   {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"  PAD:                     {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print()
# SmolLM2 uses <|im_start|> and <|im_end|> for conversation structure
# These are added to the vocabulary and the model has learned their meaning from SFT

## 4. Tokenize a Sample — Inspect the Token Ids

In [ ]:
tokenized = tokenizer(formatted, return_tensors="pt")
seq_len = tokenized["input_ids"].shape[1]

print(f"Sequence length: {seq_len} tokens")
print(f"First 20 token ids: {tokenized['input_ids'][0][:20].tolist()}")
print()

# Decode the first 20 tokens to see what they represent
print("First 20 tokens decoded:")
for i, tid in enumerate(tokenized['input_ids'][0][:20].tolist()):
    print(f"  [{i:2d}] {tid:6d} → {tokenizer.decode([tid])!r}")

## 5. Sequence Length Distribution

Training with variable-length sequences is expensive. We set a `max_seq_length` to truncate long samples.  
We want to cover most samples without truncating too many.

In [ ]:
print("Tokenizing 3k samples to measure lengths (may take ~30 seconds on CPU)...")
lengths = []
for s in samples:
    formatted = tokenizer.apply_chat_template(s["messages"], chat_template=CHATML, tokenize=False)
    n = len(tokenizer.encode(formatted))
    lengths.append(n)

import statistics
print(f"Min:    {min(lengths)}")
print(f"Max:    {max(lengths)}")
print(f"Median: {statistics.median(lengths):.0f}")
print(f"Mean:   {statistics.mean(lengths):.0f}")
print()

# What % of samples fit within common max_seq_length choices?
for cap in [512, 1024, 2048]:
    pct = sum(1 for l in lengths if l <= cap) / len(lengths) * 100
    print(f"  max_seq_length={cap:4d}: covers {pct:.1f}% of samples")

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=50, edgecolor="white", color="steelblue")
plt.axvline(1024, color="red", linestyle="--", label="max_seq_length=1024")
plt.xlabel("Sequence length (tokens)")
plt.ylabel("Count")
plt.title("smoltalk token length distribution (3k samples)")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Loss Masking — The Training Target

During SFT, we only compute loss on the **assistant** turns, not the user turns or system prompt.  
This is called **prompt masking** — the model learns to predict responses, not to repeat the question.

`SFTTrainer` (used in notebook 03) handles this automatically when you pass `messages` format.

In [ ]:
# Illustrate: show which parts of the formatted text correspond to assistant tokens
sample_msgs = sample["messages"]

print("The full formatted conversation:")
print("  [grey = prompt/user, no loss computed]")
print("  [green = assistant response, loss IS computed]")
print()

formatted_full = tokenizer.apply_chat_template(sample_msgs, chat_template=CHATML, tokenize=False)
for turn in sample_msgs:
    marker = ">>> LOSS COMPUTED HERE <<<" if turn["role"] == "assistant" else "[no loss]"
    content_preview = turn["content"][:100]
    print(f"  [{turn['role'].upper()}] {marker}")
    print(f"  '{content_preview}...'")
    print()

## Summary

What you just learned:

1. **Dataset format**: lists of `{role, content}` dicts — the standard chat format
2. **Chat template**: serializes turns into a string with special tokens (`<|im_start|>`, `<|im_end|>`)
3. **Sequence lengths**: most smoltalk samples are under 1024 tokens → use `max_seq_length=1024`
4. **Loss masking**: only the assistant tokens contribute to training loss

**Training recipe we'll use in notebook 03**:
- Model: `SmolLM2-135M` (base)
- Dataset: 2k samples from `smoltalk/smol-smoltalk`
- `max_seq_length=1024`
- 1 epoch, LoRA (r=8)

**Next**: [`03_finetune_lora.ipynb`](03_finetune_lora.ipynb) — open this on Google Colab!